# Observabilidad · Tablas de calidad de Silver

## Propósito

Crear las dos tablas donde Silver deja evidencia de lo que hizo con los datos: cuántas filas descartó al deduplicar y cuántas aisló en cuarentena.

## Tablas

| Tabla | Grano | Qué responde |
|---|---|---|
| `hd_deduplicacion_log` | una fila por ejecución y tabla Silver | ¿Cuántas filas entraron, cuántas salieron y cuántas se descartaron por duplicidad? |
| `hd_cuarentena_resumen` | una fila por ejecución, tabla Silver y motivo | ¿Cuántas filas se aislaron y por qué regla? |

## Reglas

* Se crean una sola vez (`CREATE TABLE IF NOT EXISTS`); correrlo de nuevo no borra el historial.
* Son históricas: cada ejecución agrega filas, nunca se sobrescriben.
* El detalle de cada fila en cuarentena vive en la propia tabla Silver (`estado_calidad`, `motivo_cuarentena`); aquí solo queda el conteo por ejecución.

## Origen de los datos

Los notebooks Silver escriben en estas tablas al terminar su deduplicación y su validación.

## Destino

Esquema de control: `metadata_dataco.supply_chain`.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS metadata_dataco.supply_chain.hd_deduplicacion_log (
  id_ejecucion         STRING    COMMENT 'Ejecución del pipeline que realizó la deduplicación',
  tabla_silver         STRING    COMMENT 'Tabla Silver donde se deduplicó',
  clave_deduplicacion  STRING    COMMENT 'Columna(s) que definen un registro único',
  criterio_orden       STRING    COMMENT 'Criterio para elegir el registro sobreviviente',
  filas_entrada        BIGINT    COMMENT 'Filas leídas de Bronze antes de deduplicar',
  filas_salida         BIGINT    COMMENT 'Filas que quedaron después de deduplicar',
  filas_descartadas    BIGINT    COMMENT 'Filas descartadas por duplicidad (entrada menos salida)',
  fecha_registro       TIMESTAMP COMMENT 'Momento en que se registró la fila'
)
USING DELTA

In [0]:
%sql
CREATE TABLE IF NOT EXISTS metadata_dataco.supply_chain.hd_cuarentena_resumen (
  id_ejecucion         STRING    COMMENT 'Ejecución del pipeline que aisló los registros',
  tabla_silver         STRING    COMMENT 'Tabla Silver donde se aislaron',
  motivo_cuarentena    STRING    COMMENT 'Regla incumplida que originó la cuarentena',
  filas                BIGINT    COMMENT 'Cantidad de filas aisladas por ese motivo',
  fecha_registro       TIMESTAMP COMMENT 'Momento en que se registró la fila'
)
USING DELTA

In [0]:
%sql
COMMENT ON TABLE metadata_dataco.supply_chain.hd_deduplicacion_log IS
'Auditoría de deduplicación en Silver. Grano: una fila por ejecución y tabla Silver. Histórica: solo agrega filas. Owner: ingeniería de datos.';

COMMENT ON TABLE metadata_dataco.supply_chain.hd_cuarentena_resumen IS
'Conteo de registros aislados en cuarentena por ejecución, tabla Silver y motivo. El detalle de cada registro está en la tabla Silver (estado_calidad, motivo_cuarentena). Histórica: solo agrega filas. Owner: ingeniería de datos.';

ALTER TABLE metadata_dataco.supply_chain.hd_deduplicacion_log
SET TAGS ('layer' = 'metadata', 'domain' = 'supply_chain', 'owner' = 'ingenieria_de_datos', 'frecuencia' = 'semanal');

ALTER TABLE metadata_dataco.supply_chain.hd_cuarentena_resumen
SET TAGS ('layer' = 'metadata', 'domain' = 'supply_chain', 'owner' = 'ingenieria_de_datos', 'frecuencia' = 'semanal');

In [0]:
%sql
SELECT table_name, comment
FROM metadata_dataco.information_schema.tables
WHERE table_schema = 'supply_chain'
  AND table_name IN ('hd_deduplicacion_log', 'hd_cuarentena_resumen')